# Kaggle 2×T4 GPU Orchestration

Reusable service-parallel llama.cpp infrastructure. No research/application logic.

In [ ]:
from gpu_orchestrator import *

MODEL_PATH = "/kaggle/working/model.gguf"
MTP_PATH = "/kaggle/working/model-mtp.gguf"
EMBED_MODEL_PATH = "/kaggle/working/bge-m3.gguf"

slots = make_kaggle_2xt4_slots()
config = OrchestratorConfig(
    llama_server="/kaggle/working/llama_bin/llama-server",
    checkpoint_every=10,
)

orchestrator = GPUOrchestrator(
    config=config,
    slots=slots,
    chat_args=make_gemma4_chat_args(MODEL_PATH, MTP_PATH),
    embed_args=make_embedding_args(EMBED_MODEL_PATH),
)
orchestrator.start_all()


## Application task

Replace only this section in future projects. The orchestration layer above should remain unchanged.

In [ ]:
def task_handler(task, chat_port, gpu, worker_name):
    # Your project-specific logic goes here.
    # Persist the task result before returning if you want resumability.
    return {"task": task, "chat_port": chat_port, "gpu": gpu}

tasks = [f"task-{i:03d}" for i in range(1, 101)]

scheduler = ThreadedScheduler(
    orchestrator=orchestrator,
    tasks=tasks,
    task_id=lambda x: x,
    task_handler=task_handler,
)

results = scheduler.run()


In [ ]:
# Always run cleanup at the end of the notebook.
orchestrator.stop_all()
